In [11]:
import torch
import time
import gc
from tqdm import tqdm

def _model_FPS(model, image_batch, device):
    image_batch = image_batch.to(device)
    # Warmup
    with torch.no_grad():
        for _ in tqdm(range(10), desc="Warmup"):
            _ = model(image_batch)
            if device == "cuda":
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            elif device == "mps":
                torch.mps.synchronize()
        gc.collect()
    
    # Measure FPS
    times = []
    with torch.no_grad():
        for _ in tqdm(range(50), desc="FPS measurement"):  # Run multiple iterations for more stable measurement
            start = time.time()
            _ = model(image_batch)
            if device == "cuda":
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            elif device == "mps":
                torch.mps.synchronize()
            end = time.time()
            times.append(end - start)
        gc.collect()
    
    # Calculate average FPS
    avg_time = sum(times) / len(times)
    fps = len(image_batch) / avg_time  # FPS = batch_size / time_per_batch
    return round(fps, 3)

In [12]:
%run import_local_packages.py

import os
import yaml
from src.models.core.services import MODEL_REGISTRY
from src.models.adapters.models.classifier import Classifier # Import all models into the registry
from src.models.adapters.vision_backbones import * # Import all backbones into the registry

MODEL_CONFIG_PATH = "/Users/theo.moreau/Documents/futur/src/benchmark/classification/full_shot/configs/models"

with open(os.path.join(MODEL_CONFIG_PATH, "dino_s_ft_linear_224.yaml"), "r") as f:
    model_cfg = yaml.safe_load(f)

model = MODEL_REGISTRY.build_model(model_cfg)

Using cache found in /Users/theo.moreau/.cache/torch/hub/facebookresearch_dinov2_main


In [18]:
checkpoint_path = "/Users/theo.moreau/Documents/futur/src/benchmark/classification/full_shot/checkpoints/dino_s_ft_linear_224_augmentation.pth"
weights = torch.load(checkpoint_path)

model.load_state_dict(weights)
model.to("mps")
batch = torch.randn(16, 3, 224, 224)
model.eval()

_model_FPS(model, batch, "mps")

FPS measurement: 100%|██████████| 50/50 [00:06<00:00,  8.05it/s]


129.604